In [1]:
# ensures that if we change something in .core, it sees it imediately without having to restart the kernel
# every time you run a cell, all imported modules are reloaded auomatically
%load_ext autoreload
%autoreload 2 

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
import scipy
from kneed import KneeLocator

# Our package
import neuro_lib as nlib

# SETTING SEED FOR REPRODUCIBILITY
np.random.seed(0)

relations_dict = {"linear": lambda s: s,
                "tanh": lambda s: np.tanh(5*s),
                "quadratic": lambda s: 2 * s**2 - 1}

## DEFINE TRANSFER ENTROPY AND TEST ON ARTIFICIAL DATASET

# ideas 

Assunzione,Test suggerito,Perché farlo?

Stazionarietà,Augmented Dickey-Fuller (ADF) Test,Per essere sicuro che il segnale non abbia trend o derive che invalidano la stima della densità di probabilità.

Linearità,Confronto TE vs Granger,"Calcola la TE con method='gauss' (lineare) e con method='kde' (non lineare). Se i risultati sono molto diversi, hai la prova che nel tuo segnale neurale c'è dinamica non lineare."

Ordine Markoviano,AIC/BIC o FNN,Per giustificare la scelta dell'embedding dimension m. Dimostri che non hai scelto m=1 a caso.

Significatività,Surrogati (Shuffling),Per distinguere la TE reale dal bias statistico dovuto alla lunghezza finita del segnale.


Se accade che...,La conclusione scientifica è...

TE (Gauss) ≈ TE (KDE),Il legame tra i neuroni è prevalentemente lineare. La semplicità vince.

TE (KDE) ≫ TE (Gauss),Esistono interazioni non lineari complesse (es. accoppiamento di fase) che un modello lineare non vede.

TE (Binning) è molto instabile,"Il segnale è troppo rumoroso o hai scelto pochi bin. Il binning soffre il ""bias di discretizzazione""."

TE (Copula) è la più robusta,"Le distribuzioni dei segnali non sono gaussiane (hanno code lunghe), ma la struttura di dipendenza è chiara."

A. Grafico di Robustezza al RumoreGenera un segnale AR con un lag noto. Aggiungi livelli crescenti di rumore bianco e calcola la TE con tutti i metodi.Cosa osservare: Quale metodo "crolla" per primo? Solitamente il binning è il più sensibile al rumore, mentre la Gaussian Copula è spesso la più resiliente.

B. Analisi del Tempo di CalcoloSe hai segnali molto lunghi, riporta il tempo di esecuzione.KDE: Molto preciso ma lento ($O(N^2)$).Binning/Gauss: Velocissimi.Conclusione: "Il KDE è preferibile per segmenti brevi e analisi di precisione, mentre il Binning è adatto per screening veloci su grandi dataset."

## 📋 To-Do List Definitiva

#### **Fase 1: Generazione e Pre-analisi**
* [ ] **Creazione Ground Truth:** Genera i segnali AR (lineare) e Oscillatori Accoppiati (non-lineare).
* [ ] **Test di Stazionarietà (ADF):** Verifica che i segnali siano stabili.
* [ ] **Ottimizzazione Parametri (su dati puliti e lunghi):**
    * Trova l'**Embedding Delay ($\tau$)** con il *First Minimum of MI*.
    * Trova l'**Embedding Dimension ($m$)** con il *False Nearest Neighbors (FNN)*.

#### **Fase 2: Benchmark e Lag Scanning (L'Ideale)**
* [ ] **Lag Scanning:** Esegui una scansione del lag sorgente (1-20) su dati con alto SNR e molti campioni.
* [ ] **Confronto Metodi:** Calcola la TE con **KDE, Binning, Gaussian, Gaussian Copula** per identificare il picco di informazione.


#### **Fase 3: Stress Test - Robustezza al Rumore**
* [ ] **Noise Sweep:** Fissa $N$ (es. 5000) e varia il rumore (SNR da 30dB a 0dB).
* [ ] **Analisi Errore:** Per ogni metodo, osserva a quale livello di rumore il picco di TE scompare o diventa indistinguibile dal rumore di fondo.

#### **Fase 4: Stress Test - Lunghezza del Segnale ($N$)**
* [ ] **Sample Sweep:** Fissa il rumore (basso) e varia il numero di campioni $N$ (es. 500, 1000, 2000, 5000, 10000).
* [ ] **Bias Check:** Osserva come il valore della TE "gonfia" artificialmente quando $N$ è piccolo (bias statistico).


#### **Fase 5: Validazione Statistica**
* [ ] **Permutation Test:** Applica lo shuffling della sorgente per definire la soglia di confidenza (P-value) per i risultati ottenuti nelle fasi precedenti.

#### **Fase 6: Sintesi Comparativa**
* [ ] **Report Finale:** Crea grafici che mostrano l'accuratezza (errore rispetto al lag reale) in funzione del rumore e di $N$ per tutti e 4 i metodi.

---

### 📅 Tabella di Marcia Aggiornata

| Settimana / Fase | Attività Principale | Focus Tecnico | Output Atteso |
| :--- | :--- | :--- | :--- |
| **Fase 1: Setup** | Generazione & Diagnostica | ADF, FNN, First Min MI | Parametri $m, \tau$ e stazionarietà confermata. |
| **Fase 2: Core** | Lag Scanning & Metodi | Confronto KDE, Binning, GC, Gauss | Grafico TE vs Lag (identificazione del picco). |
| **Fase 3: Rumore** | **Stress Test: Rumore** | Sweep SNR (0-30 dB) | Curve di decadimento della TE per ogni metodo. |
| **Fase 4: Dati** | **Stress Test: Campioni ($N$)** | Sweep $N$ (500-10.000) | Analisi del bias e stabilità della stima. |
| **Fase 5: Stat** | Permutation Test | Shuffling & Significatività | Barre di errore e soglie di confidenza (P < 0.05). |
| **Fase 6: Final** | Analisi Risultati | Confronto prestazioni totali | Tabella finale: "Qual è il metodo migliore?". |

---

### 💡 Un consiglio strategico per il tuo report
Quando confronterai i metodi sotto stress, probabilmente noterai che:
1.  **Gaussian Copula** sarà il più veloce e robusto al rumore.
2.  **KDE** sarà il più preciso per gli oscillatori ma "esploderà" in termini di tempo di calcolo se $N$ è troppo grande o fallirà se $N$ è troppo piccolo.
3.  **Binning** mostrerà un alto bias positivo quando $N$ è piccolo.

Presentare questi fallimenti è **più importante** che presentare solo i successi, perché dimostra che hai capito i limiti fisici dell'Information Theory.

Quale di questi due stress test ti preoccupa di più in termini di potenza di calcolo del tuo PC? (Se il KDE è lento, lo sweep su $N$ potrebbe richiedere ore).